## Apresentação 3 - Cálculo Numérico
# A MEMBRANA ELÁSTICA
**Grupo 1:** Bruno Lopes de Almeida Zuffo, 
João Victor Bozola Bosi, 
Lucas Antero Albuquerque, 
Matheus Marchi Baron, 
Natalia Yumi Watanabe, 
Victor Hugo Albertino e Silva.

---
## 1. Introdução e Parâmetros Iniciais
Neste capítulo, expandimos nosso Gêmeo Digital para englobar a dinâmica estrutural da tampa do reservatório de expansão: uma membrana elástica tensionada.
O problema é governado pela equação da onda 2D. Para evitar instabilidades numéricas oriundas da magnitude e ordem de grandeza das grandezas físicas, o sistema algébrico foi completamente adimensionalizado.

**Parâmetros Físicos Nominais:**
* **Geometria:** Membrana circular de raio $R = 0.4$ cm e espessura $e = 0.1$ mm.
* **Propriedades:** Tensão $\sigma = 200$ N/m e Densidade $\rho = 900$ kg/m³.

Neste primeiro bloco, importamos as dependências, os módulos matriciais que construímos (`functionsM.py`) e definimos os fatores de conversão para retornar as respostas físicas ao espaço temporal.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("TkAgg")
from scipy.sparse.linalg import eigsh

# Importa a função construtora base desenvolvida no arquivo auxiliar
from functionsM import BuildMatrizes_Eigen_Circular, DecomposicaoModal

# INÍCIO ----------------------------------------------------------------------------------
# Parâmetros físicos gerais e fixos do problema (Seção 3.5 do PDF)

sigma = 200.0     # Tensão (N/m)
rho = 900.0       # Densidade (kg/m^3)
e_esp = 0.1e-3    # Espessura (m) -> 0.1 mm
R = 0.4e-2        # Raio (m) -> 0.4 cm

# Fator de conversão dimensional: f_k = (w_ad / 2pi) * (1/R) * sqrt(sigma / (rho * e_esp))
fator_Hz = (1.0 / (2.0 * np.pi * R)) * np.sqrt(sigma / (rho * e_esp))

# Parâmetros adimensionais para o motor algébrico
Lx_ad = 2.0
Ly_ad = 2.0
sigma_ad = 1.0
rho_ad = 1.0
e_ad = 1.0

---
## 2. Modelagem Geométrica e Penalização (Seção 3.5.1 - Exercício 1)

O domínio de cálculo adotado é uma malha cartesiana quadrada $\hat{L}_x \times \hat{L}_y = 2 \times 2$, circunscrita à membrana circular adimensional. Para impor a condição de contorno de Dirichlet homogênea (deslocamento nulo nas bordas e fora do círculo), emprega-se o Método de Penalização via Fatiamento (Slicing) em formato CSR.

Em contraste com a manipulação elemento a elemento, esta implementação utiliza uma matriz identidade de penalidade ($Iden = 10^4 \cdot I$) para sobrescrever integralmente as linhas e colunas correspondentes aos nós restritos diretamente na matriz de rigidez ($K$). Esta técnica isola os nós externos do sistema dinâmico, forçando os seus autovalores artificiais a assumirem valores elevados e empurrando-os para o fim do espectro modal. Consequentemente, garante-se que o campo fundamental de vibração permaneça fisicamente restrito aos nós internos da membrana circular, processando a estrutura diretamente num formato de armazenamento comprimido otimizado.

In [2]:
# 3.5.1 Exercício 1 -------------------------------------------------------------------------------

N1_ex1 = 61
N2_ex1 = 61
h_ex1 = Lx_ad / (N1_ex1 - 1)

K_ex1, M_ex1 = BuildMatrizes_Eigen_Circular(N1_ex1, N2_ex1, sigma_ad, rho_ad, e_ad, h_ex1)

diag_K = K_ex1.diagonal()
mask_visual = diag_K.reshape((N2_ex1, N1_ex1))

# Exibe o grid com os pontos onde o fator de penalização (10^4) atuou rigidamente.
plt.figure(figsize=(6,6))
plt.imshow(mask_visual == 1e4, extent=[0, Lx_ad, 0, Ly_ad], origin='lower', cmap='viridis')
plt.title(f"Validação do Exercício 1: Máscara da Membrana ({N1_ex1}x{N2_ex1})\n(Região Amarela = Pontos de Contorno Restritos)")
plt.xlabel(r"$\hat{x}$")
plt.ylabel(r"$\hat{y}$")
plt.show()

---
## 3. Frequências e Modos de Vibração (Seção 3.5.1 - Exercício 2)

A dinâmica de vibrações em oscilação livre sem forçamento externo da estrutura recai obrigatoriamente no clássico problema de autovalores e autovetores generalizado:

$$K\Phi = \lambda M \Phi$$

Devido à grande dimensão das equações, a obtenção de todo o espectro completo através dos métodos lineares exatos densos (como a Fatoração QR de Francis) requer tempo e processamento de memória imensos e ineficientes. Em contrapartida, de uma perspectiva puramente aplicável à engenharia e análise estrutural, as frequências baixas de grande amplitude contêm quase que 100% da deformabilidade e preocupação em projeto (modos primordiais que causam dano e fadiga em oscilação).

Utilizamos o método iterativo de Subespaços de Krylov via rotina `scipy.sparse.linalg.eigsh` definindo o critério `'SM'` (Smallest in Magnitude) para minerar cirurgicamente apenas os 10 primeiros autovalores inferiores.

In [ ]:
# 3.5.1 Exercício 2 -------------------------------------------------------------------------------

print("\n" + "="*115)
print(f"{'RESULTADOS DO EX2 (Convergência e Modos de Vibração)':^115}")
print("="*115)

casos_ex2 = [(21, 21), (41, 41), (61, 61), (81, 81), (101, 101)]
resultados_ex2 = []

Phi_final = None
omega_final = None
f_final = None

print(f"{'Malha':<10} | " + " | ".join([f"f{i+1:<7}" for i in range(10)]))
print("-" * 115)

for Nx, Ny in casos_ex2:
    h = Lx_ad / (Nx - 1)
    
    K, M = BuildMatrizes_Eigen_Circular(Nx, Ny, sigma_ad, rho_ad, e_ad, h)
    
    # Extração de autovalores pelo método de Krylov
    lambdas, Phi = eigsh(K, k=10, M=M, which='SM')
    
    # Ordenação forçada (menor energia para maior)
    idx = np.argsort(lambdas)
    lambdas = lambdas[idx]
    Phi = Phi[:, idx]
    
    omega_ad = np.sqrt(np.abs(lambdas))
    f_Hz = omega_ad * fator_Hz
    
    resultados_ex2.append((Nx, f_Hz))
    
    linha = f"{Nx}x{Ny:<8} | " + " | ".join([f"{f:7.1f}" for f in f_Hz])
    print(linha)
    
    # Guarda as matrizes da malha mais refinada para a renderização 3D
    if Nx == 101:
        Phi_final = Phi
        omega_final = omega_ad
        f_final = f_Hz
        
print("=" * 115)

# -------------------------------------------------------------------------------------------------
# RENDERIZAÇÃO DAS SUPERFÍCIES 3D DE CADA FORMA MODAL
# -------------------------------------------------------------------------------------------------

fig = plt.figure(figsize=(20, 8))
fig.suptitle(f"Primeiros 10 Modos de Vibração - Malha Refinada {Nx}x{Ny}", fontsize=16)
fig.canvas.manager.set_window_title('Exercício 2 - Modos de Vibração')

x = np.linspace(0, Lx_ad, 101)
y = np.linspace(0, Ly_ad, 101)
X, Y = np.meshgrid(x, y)

for i in range(10):
    ax = fig.add_subplot(2, 5, i+1, projection='3d')
    
    Z = Phi_final[:, i].reshape((101, 101))
    
    surf = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.9)
    ax.set_title(f'Modo {i+1}\nf = {f_final[i]:.1f} Hz')
    
    z_limite = np.max(np.abs(Z))
    ax.set_zlim(-z_limite, z_limite)
    ax.axis('off')

plt.tight_layout()
plt.show()


                               RESULTADOS DO EX2 (Convergência e Modos de Vibração)                                
Malha      | f1       | f2       | f3       | f4       | f5       | f6       | f7       | f8       | f9       | f10     
-------------------------------------------------------------------------------------------------------------------
21x21       |  4402.7 |  6994.8 |  6994.8 |  9332.7 |  9346.9 | 10024.9 | 11548.2 | 11548.2 | 12662.7 | 12662.7
41x41       |  4445.9 |  7078.7 |  7078.7 |  9471.1 |  9486.3 | 10184.9 | 11762.1 | 11762.1 | 12924.7 | 12924.7
61x61       |  4468.9 |  7118.3 |  7118.3 |  9530.6 |  9542.8 | 10249.1 | 11841.7 | 11841.7 | 13017.2 | 13017.2
81x81       |  4474.2 |  7127.7 |  7127.7 |  9544.1 |  9557.9 | 10265.1 | 11862.1 | 11862.1 | 13041.4 | 13041.4
101x101      |  4485.1 |  7145.5 |  7145.5 |  9572.9 |  9578.3 | 10291.9 | 11894.0 | 11894.0 | 13077.1 | 13077.1


---
## 4. Decomposição Modal do Termo Forçante (Seção 3.5.1 - Exercícios 3 e 4)

*PLACEHOLDER EXERÍCIO 3* 

No Exercício 4, passamos para a área analítica de uma vibração de placa em estado de excitação. A membrana recebe um impacto ou pressão de perfil não homogêneo sobre sua área contínua, governado pelo formato parabólico estritamente geométrico da equação $Z(\hat{x}, \hat{y}) = (\hat{x}-0.5)^2 + (\hat{y}-0.5)^2$.

Ao decompor o espaço desse impacto (que está puramente numérico) usando a base de vetores dos Modos Fundamentais de Vibração obtida no Exercício 2, calculamos os coeficientes de absorção de energia ($\alpha_i$):

$$\alpha_i = \frac{(\Phi^{(i)})^T Z}{(\Phi^{(i)})^T M \Phi^{(i)}}$$

Os coeficientes identificam a proximidade paralela entre o modo de vibração que a chapa "gosta" de se deformar e como a carga a está torcendo em cima de seu referencial de centro.
A representação do gráfico logarítmico revela que modos de comportamento complexo superior sem simetria azimutal compatível perdem quase 100% da sinergia com o fator forçante, gerando resíduos próximos ao de um ruído em $\sim10^{-11}$ (vazamentos causados numéricamente nas bordas truncadas retangulares convertidas do círculo).

In [ ]:
# 3.5.1 Exercício 4 -------------------------------------------------------------------------------

Nx, Ny = 101, 101
h = Lx_ad / (Nx - 1)

K, M = BuildMatrizes_Eigen_Circular(Nx, Ny, sigma_ad, rho_ad, e_ad, h)
lambdas, Phi = eigsh(K, k=10, M=M, which='SM')

idx = np.argsort(lambdas)
lambdas = lambdas[idx]
Phi = Phi[:, idx]

x = np.linspace(0, Lx_ad, Nx)
y = np.linspace(0, Ly_ad, Ny)
X, Y = np.meshgrid(x, y)

Z_grid = (X - 0.5)**2 + (Y - 0.5)**2
Z = Z_grid.flatten()

# Executa a decomposição ortogonal criada
alpha = DecomposicaoModal(Phi, M, Z)

print("\n" + "="*70)
print(f"{'DECOMPOSIÇÃO MODAL DO TERMO FORÇANTE (Malha 101x101)':^70}")
print("="*70)
for i, a in enumerate(alpha):
    print(f"Modo {i+1:2d}: alpha = {a:.6e}")
print("="*70)

modos = np.arange(1, len(alpha)+1)
alpha_abs = np.abs(alpha)

plt.figure(figsize=(10,5))
plt.bar(modos, alpha_abs, color="#483D8B")
plt.xlabel('Modo de Vibração')
plt.ylabel(r'$|\alpha_i|$')
plt.title('Módulo dos Coeficientes Modais (Afinidade Força x Modo)')
plt.xticks(modos)
plt.yscale('log')
plt.grid(axis='y', alpha=0.3)
plt.show()


                 DECOMPOSIÇÃO MODAL DO TERMO FORÇANTE                 
Modo  1: alpha = -6.016152e+01
Modo  2: alpha = -4.659210e+01
Modo  3: alpha = 4.071486e+00
Modo  4: alpha = -3.921398e-12
Modo  5: alpha = -1.395994e-12
Modo  6: alpha = -4.449157e+01
Modo  7: alpha = -6.950234e-02
Modo  8: alpha = 1.881073e-02
Modo  9: alpha = -2.088430e+01
Modo 10: alpha = -1.468718e+01


---
## 5. Energia Elástica Média e Curvas de Ressonância (Seção 3.5.1 - Exercício 5)

Como elemento de projeto, modelamos a resposta estrutural no domínio da frequência frente a uma oscilação contínua forçada (agora amortecida).
O problema é definido pelo amortecimento introduzido linearmente proporcional à massa ($M$). Realizamos uma varredura sobre um espectro contínuo de excitações (com velocidades de pulsação controladas de 0.5 a 100 $\hat{\omega}_*$) e medimos a energia elástica concentrada pelo elastômero sob três variações de fator $\beta$.

* **Picos Perigosos de Ressonância:** Quando a excitação do pulso injetado coincide fisicamente com as características adimensionais intrínsecas (os modos axiais que vimos acima), a matriz absorve força infinitamente, deformando sem freios.
* **Amortecimento Severo:** Para $\beta = 1.0$ (verde), as espículas param de se alinhar de forma infinita e dissipam energia rapidamente em atrito sem permitir vibração letal. O pico perde a agulha de alta periculosidade.
* **Fenômeno de Truncamento Algébrico:** O que parece ser uma estabilidade súbita e queda radical por volta da métrica $30$ de pulso ocorre estritamente pela deficiência da amostra. Nossa malha possui truncamento definido pelo extrator em $k=200$. O modelo é incapaz matematicamente de absorver frequências em modos além das primeiras duas centenas que ele próprio tem guardado da matriz originária.

In [ ]:
# 3.5.1 Exercício 5 -------------------------------------------------------------------------------

print("\n" + "="*70)
print(f"{'CÁLCULO DA ENERGIA ELÁSTICA MÉDIA (Curva de Ressonância - Malha 101x101)':^70}")
print("="*70)

num_modos_ex5 = 200
lambdas_ex5, Phi_ex5 = eigsh(K, k=num_modos_ex5, M=M, which='SM')

idx_ex5 = np.argsort(lambdas_ex5)
lambdas_ex5 = lambdas_ex5[idx_ex5]
Phi_ex5 = Phi_ex5[:, idx_ex5]

# Frequências naturais para este bloco
omega_i_ex5 = np.sqrt(np.abs(lambdas_ex5))

alpha_ex5 = DecomposicaoModal(Phi_ex5, M, Z)

# Definindo o vetor de frequências de excitação (escala logarítmica de 0.5 a 100)
omega_star = np.logspace(np.log10(0.5), np.log10(100), 2000)

# Valores de amortecimento solicitados no exercício
betas = [0.01, 0.1, 1.0]

plt.figure(figsize=(10, 6))

for beta in betas:
    Ae = np.zeros_like(omega_star)
    
    for k_idx, w_star in enumerate(omega_star):
        # Calculando os coeficientes c_i para a frequência w_star atual
        denominador = np.sqrt((-w_star**2 + omega_i_ex5**2)**2 + (beta * w_star)**2)
        c_i = alpha_ex5 / denominador
        
        # Calculando a energia elástica média
        Ae[k_idx] = 0.25 * np.sum((c_i**2) * (omega_i_ex5**2))
        
    # Plotando a curva para o beta atual
    plt.loglog(omega_star, Ae, label=rf'$\beta$ = {beta}')

# Configurações do gráfico para replicar a Figura do PDF
plt.xlabel('Frequência do Termo Forçante ($\\hat{\\omega}_*$)')
plt.ylabel('Energia Média ($A_e$)')
plt.title('Energia Elástica Média da Membrana vs Frequência (Adimensional)')
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.xlim(0.5, 100)
plt.ylim(bottom=1e-1) 
plt.tight_layout()
plt.show()


       CÁLCULO DA ENERGIA ELÁSTICA MÉDIA (Curva de Ressonância)       
